Import libraries

In [ ]:
#!pip install tensorflow

In [41]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, models
import numpy as np

Load splits.csv

In [42]:
df = pd.read_csv("../data/splits.csv")

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

Parameters (MATCH TEAM SETTINGS)

In [43]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

Function to load images

In [44]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img, label

Create TensorFlow datasets

In [45]:
def create_dataset(df):
    paths = df["filepath"].values
    labels = df["class_idx"].values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda x, y: load_image(x, y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return ds

train_ds = create_dataset(train_df)
val_ds   = create_dataset(val_df)
test_ds  = create_dataset(test_df)

C:\Users\TUF\AppData\Local\Temp\ipykernel_13404\2398227592.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["filepath"] = train_df["filepath"].str.strip()
C:\Users\TUF\AppData\Local\Temp\ipykernel_13404\2398227592.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df["filepath"]   = val_df["filepath"].str.strip()
C:\Users\TUF\AppData\Local\Temp\ipykernel_13404\2398227592.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_inde

BASE_DIR: D:\1. Projects\Trash Classification System\Trash-Classification-System


FileNotFoundError: File not found in any folder: data\raw\images\shoes\real_world\Image_182.png

Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

Build MobileNetV3-Large model

In [ ]:
base_model = MobileNetV3Large(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

Compile

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Fine-tuning (IMPORTANT)

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, validation_data=val_ds, epochs=5)

Test accuracy

In [ ]:
loss, acc = model.evaluate(test_ds)
print("Test Accuracy:", acc)

Save model

In [ ]:
model.save("models/mobilenetv3_member3.h5")